# Quantum Phase Estimation

QPE estimates the eigenvalue of a unitary operator.
Here we estimate the phase of the T gate (eigenvalue e^(2\u03c0i/8)).

In [ ]:
import cirq
import numpy as np

## QPE Circuit (3 precision qubits)

Prepare |1\u27e9 on target, Hadamards on precision, controlled unitaries, inverse QFT.

In [ ]:
def qpe_circuit(precision, target, unitary_power=1):
    n = len(precision)
    ops = [cirq.X(target)]
    ops.extend(cirq.H(q) for q in precision)
    for i in range(n):
        power = unitary_power * 2 ** (n - 1 - i)
        ops.append(cirq.CPowGate(exponent=power / 8)(precision[i], target))
    for i in range(n // 2):
        ops.append(cirq.SWAP(precision[i], precision[n - 1 - i]))
    for i in range(n - 1, -1, -1):
        for j in range(i + 1, n):
            k = j - i
            ops.append(cirq.CZPowGate(exponent=-1.0 / (2**k))(precision[j], precision[i]))
        ops.append(cirq.H(precision[i]))
    return cirq.Circuit(ops)

precision = cirq.LineQubit.range(3)
target = cirq.LineQubit(3)
print(qpe_circuit(precision, target))

## Estimate T gate phase (theta = 1/8 = 0.125)

In [ ]:
sim = cirq.Simulator()
result = sim.simulate(qpe_circuit(precision, target))
sv = result.final_state_vector

target_dim = 2
prec_dim = 8
print("Precision register amplitudes:")
for i in range(prec_dim):
    amp = sum(abs(sv[i * target_dim + t]) ** 2 for t in range(target_dim))
    if amp > 1e-6:
        print(f"  |{i:03b}\u27e9: theta ~ {i/prec_dim:.4f} (phase ~ {i/prec_dim * 2 * np.pi:.4f} rad)")

print("\nExpected: theta = 1/8 = 0.125")

## Measurement statistics

In [ ]:
meas = qpe_circuit(precision, target) + cirq.measure(*precision, key="prec")
result = sim.run(meas, repetitions=1000)
counts = result.histogram(key="prec")
for k in sorted(counts):
    print(f"  |{k:03b}\u27e9: {counts[k]} shots -> theta = {k/8:.4f}")